# 01 · Variant-stability visualization

Visualize supplied mutation-effect predictions. Confirm the source model and ΔΔG sign convention before interpreting stabilization. Positions use the input numbering, not an assumed 453-residue index.

## Configuration

Use `config.local.json` as described in [the workflow guide](../docs/WORKFLOW.md). Inputs must be authorized for local analysis. Outputs are cleared in version control; no research result is asserted by this notebook.

In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'analysis/interactions.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Start Jupyter from this repository')
sys.path.insert(0, str(root))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from analysis.notebook_setup import load_config, load_manifest
config = load_config()
out = config['output_dir']


## Validate and reshape predictions

In [ ]:
predictions = pd.read_csv(config['variant_predictions'])
required = ['mutation', 'position', 'ddG_pred']
if not set(required) <= set(predictions):
    raise ValueError(f'Prediction table requires {required}')
if predictions[required].isna().any().any():
    raise ValueError('Missing prediction value')
if predictions.duplicated(['mutation', 'position']).any():
    raise ValueError('Duplicate mutation/position prediction')
predictions['ddG_pred'] = pd.to_numeric(predictions['ddG_pred'], errors='raise')
if not np.isfinite(predictions.ddG_pred).all():
    raise ValueError('Non-finite prediction')
heatmap = predictions.pivot(index='mutation', columns='position', values='ddG_pred')
heatmap

## Plot predicted effects

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(heatmap, cmap='RdBu_r', center=0, ax=ax,
            cbar_kws={'label': 'Predicted ΔΔG (source convention)'})
ax.set(xlabel='Position (input numbering)', ylabel='Mutation', title='Predicted variant effects')
fig.tight_layout()
fig.savefig(out / 'variant_predictions.svg')
plt.show()